In [ ]:
import os
import json
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# --- Configuration ---
# MiniCPM-V 2.6 (8B)
# TinyChart
# Claude 3.5 Sonnet
# Swap this ID to test Llama or InternVL later
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  

TEST_JSON_PATH = "extracted_chart_specs_test_300.json"
TEST_IMG_DIR = "./test_images_300"
OUTPUT_FILE = f"baseline{MODEL_ID}.json"

def run_baseline_inference():
    print(f"Loading Base Model {MODEL_ID} into VRAM...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        device_map="auto",
        dtype=torch.bfloat16
    )
    model.eval()

    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)

    predictions_record = []

   schema_prompt = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {
    "type": "line/bar/scatter/pie/area/box/histogram", 
    "sub": "standard/vertical/horizontal/bubble/stacked/unstacked/density/donut"
  },
  "legend": {
    "on": true/false, 
    "title": "Legend title or null"
  },
  "axes": [
    {
      "name": "x_axis or y_axis", 
      "lbl": "Axis label or null", 
      "scl": "linear/log/categorical", 
      "dom": ["min_val", "max_val"]
    }
  ],
  "ser": [
    {
      "name": "Series Name",
      "color": "#hexcode or null",
      "y_ref": "y_axis",
      "data": [["x_val", "y_val"]],
      "trend": {
        "global_trend": "stable/monotonic_increase/monotonic_decrease/fluctuating/peak_then_drop/valley_then_rise",
        "rate_of_change": "stable/slow/moderate/rapid",
        "intervals": [{"x_range": ["x1", "x2"], "trend": "String"}]
      },
      "stats": {
        "max_value": 0.0,
        "max_at_the_ref_point": "x_val",
        "min_value": 0.0,
        "min_at_the_ref_point": "x_val",
        "threshold_coverage": {"above_mean": true, "all_positive": true, "hits_zero": false},
        "crossing_points": [{"series_pair": ["Series A", "Series B"], "ref": "x_val", "value": 0.0}]
      }
    }
  ],
  "rel": [
    {
      "relation_type": "String (e.g., global_maximums_y_axis)", 
      "description": "Natural language description of the relationship.", 
      "ranking": ["Series A", "Series B"]
    }
  ]
}

Format Rules:
- For Area charts, 'data' is ["x_val", "y_max", "y_min"].
- For Box plots, 'data' is ["x_val", "min", "q1", "median", "q3", "max"].
- For Histograms, 'data' is ["bin_start", "bin_end", "height"].
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

    for index, gt_item in enumerate(ground_truth_data, start=1):
        chart_id = gt_item.get("id")
        ctype = gt_item.get("chart_type")
        img_path = os.path.join(TEST_IMG_DIR, f"{chart_id}.png")
        
        print(f"Zero-Shot Eval [{index}/{len(ground_truth_data)}] - {ctype} (ID: {chart_id})")
        if not os.path.exists(img_path): continue

        messages = [
            {"role": "system", "content": "You output strictly valid JSON with no markdown formatting or extra text."},
            {"role": "user", "content": [
                {"type": "image", "image": img_path, "max_pixels": 768 * 768},
                {"type": "text", "text": schema_prompt}
            ]}
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=4096,
                temperature=0.1,  
                do_sample=False
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        prediction_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        
        # Clean up Markdown backticks if the model ignored the system prompt
        if prediction_text.startswith("```json"):
            prediction_text = prediction_text.replace("```json", "").replace("```", "").strip()

        predictions_record.append({
            "id": chart_id,
            "chart_type": ctype,
            "predicted_text": prediction_text
        })

    with open(OUTPUT_FILE, 'w', encoding="utf-8") as f:
        json.dump(predictions_record, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Baseline predictions saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    run_baseline_inference()